# Notebook 03 — Experiment 2 orchestration comparison

Canonical analysis entry point for **Experiment 2: AaT vs Plan-Execute (and optional Verified PE / Hybrid)**.

Scope:

| Cell | Label | Orchestration | Directory |
|---|---|---|---|
| B | AaT MCP-baseline | `agent_as_tool` | `benchmarks/cell_B_mcp_baseline/` |
| Y | PE | `plan_execute` | `benchmarks/cell_Y_plan_execute/` |
| Z | Verified PE (Hybrid) | `verified_pe` | `benchmarks/cell_Z_hybrid/` |

Cell B is **shared** with Experiment 1 — it's the AaT MCP-baseline. That's why it shows up in both Notebook 02 (where it's the transport-overhead anchor) and Notebook 03 (where it's the AaT orchestration baseline). The `CONTRIBUTING_EXPERIMENTS="exp1_mcp_overhead,exp2_orchestration"` tag on Cell B's config makes this explicit.

Runner output shape (per-scenario JSON written by `run_experiment.sh` + AOB / repo-local runners):

- `success` — bool, true if no failed_steps
- `failed_steps` — list of step indices (or descriptors) that failed
- `history` — list of step entries with tool calls, responses, errors, latencies
- `plan` — the initial plan (for PE / Verified PE)
- `answer` — final answer string
- per-file latency in `latencies.jsonl` under the run dir

Until all three cells have committed captures, this notebook runs in **preflight mode** and only writes `results/metrics/notebook03_cell_availability.csv`.

In [ ]:
import json
import platform
from pathlib import Path
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "benchmarks").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path")

REPO_ROOT = find_repo_root(Path.cwd())
BENCHMARKS_DIR = REPO_ROOT / "benchmarks"
RESULTS_METRICS_DIR = REPO_ROOT / "results" / "metrics"
RESULTS_FIGURES_DIR = REPO_ROOT / "results" / "figures"
RESULTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:   {REPO_ROOT}")
print(f"Python:      {platform.python_version()}")
print(f"pandas:      {pd.__version__}")
print(f"numpy:       {np.__version__}")
print(f"matplotlib:  {matplotlib.__version__}")

## Cells scanned

In [ ]:
CELL_SPECS = [
    {"cell": "B", "label": "AaT (MCP baseline)", "path": BENCHMARKS_DIR / "cell_B_mcp_baseline"},
    {"cell": "Y", "label": "Plan-Execute",        "path": BENCHMARKS_DIR / "cell_Y_plan_execute"},
    {"cell": "Z", "label": "Verified PE (Hybrid)","path": BENCHMARKS_DIR / "cell_Z_hybrid"},
]

SUMMARY_FIELDS = [
    "experiment_family", "experiment_cell", "orchestration_mode", "mcp_mode",
    "scenario_set_name", "scenario_set_hash", "model_id",
    "run_status", "scenarios_attempted", "scenarios_completed",
    "success_rate", "failure_count",
    "wall_clock_seconds_total",
    "latency_seconds_mean", "latency_seconds_p50", "latency_seconds_p95",
    "tool_call_count_total", "tool_call_count_mean", "tool_error_count",
    "mcp_latency_seconds_mean", "mcp_latency_seconds_p95",
    "tool_latency_seconds_mean",
    "wandb_run_url",
    "judge_score_mean", "judge_score_p50", "judge_score_p95",
    "judge_pass_rate",
]

def _load_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None

def _count_jsonl_rows(path: Path) -> int:
    if path is None or not path.exists():
        return 0
    with path.open() as handle:
        return sum(1 for line in handle if line.strip())

def _scenario_jsons(run_dir: Path):
    if not run_dir.exists():
        return []
    # Per-scenario runs are JSON files in run_dir that are not meta / summary.
    return sorted(
        p for p in run_dir.glob("*.json")
        if p.name not in {"meta.json", "summary.json", "config.json"}
    )

def scan_cell(cell_spec: dict) -> dict:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()]) if raw_dir.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None

    summary = _load_json(cell_dir / "summary.json")
    config = _load_json(cell_dir / "config.json")
    latencies_path = latest_run / "latencies.jsonl" if latest_run else None
    scenario_files = _scenario_jsons(latest_run) if latest_run else []

    row = {
        "cell": cell_spec["cell"],
        "label": cell_spec["label"],
        "cell_dir": str(cell_dir.relative_to(REPO_ROOT)),
        "config_present": (cell_dir / "config.json").exists(),
        "summary_present": (cell_dir / "summary.json").exists(),
        "raw_run_count": len(run_dirs),
        "latest_run_id": latest_run.name if latest_run else "",
        "latency_rows": _count_jsonl_rows(latencies_path) if latencies_path else 0,
        "scenario_json_count": len(scenario_files),
    }
    for field in SUMMARY_FIELDS:
        row[field] = summary.get(field) if summary else None
    return row

availability_df = pd.DataFrame(scan_cell(spec) for spec in CELL_SPECS)
availability_df

In [ ]:
availability_path = RESULTS_METRICS_DIR / "notebook03_cell_availability.csv"
availability_df.to_csv(availability_path, index=False)
print(f"Wrote {availability_path.relative_to(REPO_ROOT)}")

display_cols = [
    "cell", "label",
    "config_present", "summary_present", "raw_run_count",
    "latest_run_id", "latency_rows", "scenario_json_count",
    "orchestration_mode", "mcp_mode",
    "success_rate", "latency_seconds_p50", "latency_seconds_p95",
    "tool_call_count_mean", "tool_error_count",
    "judge_score_mean", "judge_pass_rate",
]
display(availability_df[display_cols])

## Readiness gate

Notebook 03 only claims orchestration comparison findings once **all three cells** have:

- a committed `config.json`
- a committed `summary.json`
- at least one raw run directory with ≥1 per-scenario JSON

Note: the Verified PE runs in `docs/validation_log.md` (Slurm jobs `8850716` / `8851966`) predate the benchmark-wrapper success-accounting fix and the JSON error-payload fix from Codex's 2026-04-20 finding. Those need to rerun before this notebook can produce a real orchestration comparison — otherwise `success` and `failed_steps` counts will be contaminated.

In [ ]:
availability_df["analysis_ready"] = (
    availability_df["config_present"]
    & availability_df["summary_present"]
    & (availability_df["raw_run_count"] > 0)
    & (availability_df["scenario_json_count"] > 0)
)

ready_cells = availability_df.loc[availability_df["analysis_ready"], "cell"].tolist()
missing_cells = availability_df.loc[~availability_df["analysis_ready"], "cell"].tolist()

print(f"Ready cells:   {ready_cells if ready_cells else 'none'}")
print(f"Missing cells: {missing_cells if missing_cells else 'none'}")

analysis_ready = len(missing_cells) == 0
if not analysis_ready:
    print("\nScaffold mode: waiting on committed scenario JSONs for Cells B, Y, Z.")

## Per-scenario extraction

For each cell's latest run, extract per-scenario outcomes from the scenario JSONs:

- `success` / `failed_steps` from the top level
- `history` length (proxy for tool-call depth)
- `tool_errors` count (scan history for `error` payloads)

`failed_steps` is a list; `len(failed_steps)` is the scenario-level failure count. A scenario with `success=True` but non-empty `failed_steps` indicates a recovery path (relevant for Verified PE replan behavior).

Judge scores live in `results/metrics/scenario_scores.jsonl` per `#17` — joined back into the comparison when present.

In [ ]:
def load_scenario_records(cell_spec: dict) -> pd.DataFrame:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
    if not run_dirs:
        return pd.DataFrame()
    latest_run = run_dirs[-1]
    rows = []
    for scenario_file in _scenario_jsons(latest_run):
        try:
            data = json.loads(scenario_file.read_text())
        except json.JSONDecodeError:
            continue
        history = data.get("history") or []
        tool_errors = 0
        for step in history:
            if isinstance(step, dict):
                # runner shape: step has 'success' + optional 'error' + 'response'
                if step.get("success") is False:
                    tool_errors += 1
                # belt-and-suspenders: also detect JSON-error-payload masking,
                # which the wrapper fix must catch but older runs may have missed.
                resp = step.get("response")
                if isinstance(resp, dict) and resp.get("error"):
                    tool_errors += 1
        scenario_id = data.get("scenario", {}).get("id") if isinstance(data.get("scenario"), dict) else None
        if scenario_id is None:
            scenario_id = scenario_file.stem
        rows.append({
            "cell": cell_spec["cell"],
            "label": cell_spec["label"],
            "run_id": latest_run.name,
            "scenario_id": scenario_id,
            "scenario_file": scenario_file.name,
            "success": bool(data.get("success")),
            "failed_steps_count": len(data.get("failed_steps") or []),
            "history_length": len(history),
            "tool_error_count": tool_errors,
            "answer_length_chars": len(data.get("answer") or ""),
            "replans_used": (data.get("verification") or {}).get("replans_used") if isinstance(data.get("verification"), dict) else None,
        })
    return pd.DataFrame(rows)

def load_judge_scores() -> pd.DataFrame:
    scores_path = RESULTS_METRICS_DIR / "scenario_scores.jsonl"
    if not scores_path.exists():
        return pd.DataFrame()
    rows = []
    with scores_path.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(rows)

if analysis_ready:
    scenario_df = pd.concat(
        [load_scenario_records(spec) for spec in CELL_SPECS],
        ignore_index=True,
    )
    judge_df = load_judge_scores()
    print(f"Scenario rows: {len(scenario_df)}")
    print(f"Judge rows:    {len(judge_df)}")
    display(scenario_df.head(10))
else:
    scenario_df = pd.DataFrame()
    judge_df = pd.DataFrame()
    print("Skipping scenario extraction — cells missing captures.")

## Orchestration comparison table

Aggregate per cell (mean / distribution across scenarios):

- **success rate** — fraction where `success=True`
- **mean failed steps** — average `len(failed_steps)` across scenarios
- **mean history length** — average tool-call depth (useful for comparing PE's plan-then-execute depth vs AaT's free-form ReAct depth)
- **mean tool-error count** — catches both `step.success=False` and JSON-error-payload masking
- **mean replans used** — Verified PE only; zero for AaT / plain PE

Judge pass rate joined when available (per `#17`).

In [ ]:
if analysis_ready and not scenario_df.empty:
    agg = (
        scenario_df.groupby(["cell", "label"], as_index=False)
        .agg(
            scenarios=("scenario_id", "count"),
            success_rate=("success", "mean"),
            mean_failed_steps=("failed_steps_count", "mean"),
            mean_history_length=("history_length", "mean"),
            mean_tool_errors=("tool_error_count", "mean"),
            mean_replans=("replans_used", "mean"),
        )
    )

    if not judge_df.empty and "cell" in judge_df.columns:
        judge_agg = (
            judge_df.groupby("cell", as_index=False)
            .agg(
                judge_scenarios=("scenario_id", "count"),
                judge_mean_score=("overall_score", "mean") if "overall_score" in judge_df.columns else ("scenario_id", "count"),
            )
        )
        agg = agg.merge(judge_agg, on="cell", how="left")

    comparison_path = RESULTS_METRICS_DIR / "notebook03_orchestration_comparison.csv"
    agg.to_csv(comparison_path, index=False)
    print(f"Wrote {comparison_path.relative_to(REPO_ROOT)}")
    display(agg)
else:
    print("Orchestration comparison deferred — need scenario captures across all three cells.")

## Figure — success rate + latency by orchestration

Paper-facing plot. Two panels: success rate (bar) and p50 latency (bar with p95 cap), sorted B → Y → Z to keep the "transport-constant, orchestration-varying" reading order.

In [ ]:
if analysis_ready and not scenario_df.empty and not availability_df.empty:
    summary = availability_df.set_index("cell").loc[["B", "Y", "Z"]]

    sr = scenario_df.groupby("cell")["success"].mean().reindex(["B", "Y", "Z"])
    p50 = summary["latency_seconds_p50"].astype(float).values
    p95 = summary["latency_seconds_p95"].astype(float).values
    labels = summary["label"].tolist()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    colors = ["#f58518", "#4c78a8", "#e45756"]

    ax1.bar(labels, sr.values, color=colors, edgecolor="black", linewidth=0.5)
    ax1.set_ylim(0, 1.05)
    ax1.set_title("Success rate by orchestration")
    ax1.set_ylabel("Fraction of scenarios with success=True")
    ax1.grid(axis="y", alpha=0.3)

    bars = ax2.bar(labels, p50, color=colors, edgecolor="black", linewidth=0.5)
    for x, mid, high in zip(bars, p50, p95):
        if np.isnan(mid) or np.isnan(high):
            continue
        cx = x.get_x() + x.get_width() / 2
        ax2.plot([cx, cx], [mid, high], color="black", linewidth=1)
        ax2.plot([cx - 0.12, cx + 0.12], [high, high], color="black", linewidth=1)
    ax2.set_title("Latency by orchestration (p50 bar, p95 cap)")
    ax2.set_ylabel("End-to-end latency (seconds)")
    ax2.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    figure_path = RESULTS_FIGURES_DIR / "notebook03_orchestration_comparison.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {figure_path.relative_to(REPO_ROOT)}")
else:
    print("Figure deferred — need all three cells captured.")

## Failure pattern — where does each orchestration break?

When `failed_steps` or `tool_error_count` is non-zero, where in the trajectory does it happen? For PE / Verified PE, step index tells us whether failures cluster at plan formation vs execution vs verification. For AaT, step index corresponds to ReAct-loop iteration.

This section is a placeholder until scenario captures land. When they do, the right artifact is a failure-step histogram per cell.

In [ ]:
if analysis_ready and not scenario_df.empty:
    # Simple per-cell breakdown: what fraction of scenarios had any failures.
    failure_breakdown = scenario_df.assign(
        any_failure=(scenario_df["failed_steps_count"] > 0) | (scenario_df["tool_error_count"] > 0),
        recovered=(scenario_df["failed_steps_count"] > 0) & scenario_df["success"],
    ).groupby(["cell", "label"], as_index=False).agg(
        scenarios=("scenario_id", "count"),
        any_failure=("any_failure", "sum"),
        recovered=("recovered", "sum"),
    )
    failure_breakdown["recovery_rate"] = (
        failure_breakdown["recovered"] / failure_breakdown["any_failure"].where(failure_breakdown["any_failure"] > 0)
    ).fillna(0)

    breakdown_path = RESULTS_METRICS_DIR / "notebook03_failure_breakdown.csv"
    failure_breakdown.to_csv(breakdown_path, index=False)
    print(f"Wrote {breakdown_path.relative_to(REPO_ROOT)}")
    display(failure_breakdown)
else:
    print("Failure breakdown deferred — need scenario captures.")

## What changes later

Forward-looking deltas when real Experiment 2 captures land:

- per-step latency extraction from scenario JSONs (each `history` entry carries a latency if the runner records it)
- judge score join (`#17` — once `results/metrics/scenario_scores.jsonl` has entries from Maverick-17B)
- replan behavior distribution for Cell Z (histogram of `replans_used` — does verification actually help or just add cost?)
- AaT vs PE qualitative analysis — does PE need fewer tool calls because the plan is committed upfront?

**Dependencies worth tracking:**

- `#17` judge wiring (Akshat) — feeds judge score join
- `#23` / `#24` branch merges — provides Cell Z (Verified PE) captures
- `#115` server-hardening merge — stabilizes Cell Y's tool-error baseline (tool-error count is load-bearing for the "recovery" analysis above)